# MLP Baseline — MPP (маленькая модель для тестирования)

Маленькая версия MLP baseline для быстрой проверки всех этапов:
- Загрузка данных и построение словарей
- Создание датасета и коллатора
- Инициализация MLPMaskedPlayerModel
- Обучение через HF Trainer
- Оценка на валидации (top-1, top-3 accuracy)
- Сохранение лучшей модели

**Параметры маленькой модели**: embed_size=32, num_layers=1, forward_expansion=2

In [ ]:
import sys
from pathlib import Path

# Определяем корень проекта
ROOT = Path(".").resolve()
while ROOT.name and not (ROOT / "models").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT)
print("Device:", device)

## 1. Данные и словари

In [ ]:
from data.preprocessing import preprocess_raw_csv, build_vocab_mappings

raw_path = ROOT / "dataset" / "data_with_dates.csv"
sample_path = ROOT / "notebooks" / "baselines" / "MLP_baseline" / "train_sample_raw.csv"
output_dir = str(ROOT / "notebooks" / "baselines" / "MLP_baseline" / "processed")

df_raw = pd.read_csv(raw_path)
df_raw.to_csv(sample_path, index=False)
df = preprocess_raw_csv(str(sample_path), output_dir)
vocab = build_vocab_mappings(df, output_dir)

print("Матчей (уникальных match_id):", df["match_id"].nunique())
print("players_vocab_size:", vocab["players_vocab_size"])
print("player_pad_token_id:", vocab["player_pad_token_id"])
print("player_mask_token_id:", vocab["player_mask_token_id"])

## 2. Датасет и коллатор

In [ ]:
from data.dataset import MatchDatasetMPP
from data.collator import DataCollatorMPP
from torch.utils.data import Subset

max_seq_length = 36
seed = 42

ds_full = MatchDatasetMPP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=max_seq_length,
    player_pad_token_id=vocab["player_pad_token_id"],
    team_pad_token_id=vocab["team_pad_token_id"],
    position_pad_token_id=25,
)

n = len(ds_full)
n_val = max(1, int(n * 0.05))
n_train = n - n_val

np.random.seed(seed)
indices = np.random.permutation(n)
train_dataset = Subset(ds_full, indices[:n_train].tolist())
eval_dataset = Subset(ds_full, indices[n_train:].tolist())

collator = DataCollatorMPP(
    player_mask_token_id=vocab["player_mask_token_id"],
    mask_percentage=0.25,
)

print("Train матчей:", n_train, "Eval матчей:", n_val)

## 3. Маленькая MLP модель

In [ ]:
from baselines.MLP_baseline.MLP_baseline import MLPMaskedPlayerModel
from models.pretrain import MaskedPlayerModel

# --- Маленькая модель для быстрого тестирования ---
EMBED_SIZE = 32
NUM_LAYERS = 1
FORWARD_EXPANSION = 2
DROPOUT = 0.05

model = MLPMaskedPlayerModel(
    embed_size=EMBED_SIZE,
    num_layers=NUM_LAYERS,
    forward_expansion=FORWARD_EXPANSION,
    dropout=DROPOUT,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"MLP Baseline (маленькая) параметров: {n_params:,} (trainable: {n_trainable:,})")

# Для сравнения — основная модель с теми же размерами
model_ref = MaskedPlayerModel(
    embed_size=EMBED_SIZE,
    num_layers=NUM_LAYERS,
    heads=2,
    forward_expansion=FORWARD_EXPANSION,
    dropout=DROPOUT,
    form_stats_size=39,
    players_vocab_size=vocab["player_pad_token_id"],
    teams_vocab_size=vocab["team_pad_token_id"],
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
n_ref = sum(p.numel() for p in model_ref.parameters())
print(f"Transformer (маленький) параметров: {n_ref:,}")
del model_ref

## 4. Обучение

In [ ]:
from training.trainer import build_training_args, build_trainer
from training.metrics import compute_metrics_mpp

output_dir = str(ROOT / "notebooks" / "baselines" / "MLP_baseline" / "output_little")

training_config = {
    "output_dir": output_dir,
    "num_train_epochs": 50,
    "per_device_train_batch_size": 64,
    "per_device_eval_batch_size": 64,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "warmup_ratio": 0.0,
    "lr_scheduler_type": "linear",
    "logging_steps": 20,
    "eval_strategy": "steps",
    "eval_steps": 20,
    "save_strategy": "steps",
    "save_steps": 200,
    "save_total_limit": 2,
    "report_to": "none",
    "seed": seed,
    "load_best_model_at_end": True,
    "metric_for_best_model": "accuracy_top1",
    "greater_is_better": True,
}

train_args = build_training_args(training_config)
trainer = build_trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics_mpp,
    data_collator=collator,
)

print("Trainer готов.")

In [ ]:
best_model_dir = Path(output_dir) / "best_model"

try:
    trainer.train()
finally:
    trainer.save_model(str(best_model_dir))
    print(f"Модель сохранена: {best_model_dir}")

if trainer.state.log_history:
    pd.DataFrame(trainer.state.log_history).to_csv(
        Path(output_dir) / "metrics.csv", index=False
    )
    print("Метрики сохранены в metrics.csv")

## 5. Оценка на валидации

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

eval_results = trainer.evaluate()
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

## 6. Визуализация метрик

In [ ]:
import matplotlib.pyplot as plt

metrics_df = pd.DataFrame(trainer.state.log_history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
train_loss = metrics_df.dropna(subset=["loss"])
eval_loss = metrics_df.dropna(subset=["eval_loss"])
axes[0].plot(train_loss["step"], train_loss["loss"], label="train")
axes[0].plot(eval_loss["step"], eval_loss["eval_loss"], label="eval")
axes[0].set_xlabel("step")
axes[0].set_ylabel("loss")
axes[0].set_title("Loss")
axes[0].legend()

# Top-1 accuracy
eval_acc = metrics_df.dropna(subset=["eval_accuracy_top1"])
axes[1].plot(eval_acc["step"], eval_acc["eval_accuracy_top1"])
axes[1].set_xlabel("step")
axes[1].set_ylabel("accuracy")
axes[1].set_title("Eval Top-1 Accuracy")

# Top-3 accuracy
axes[2].plot(eval_acc["step"], eval_acc["eval_accuracy_top3"])
axes[2].set_xlabel("step")
axes[2].set_ylabel("accuracy")
axes[2].set_title("Eval Top-3 Accuracy")

plt.tight_layout()
plt.savefig(Path(output_dir) / "metrics_plot.png", dpi=100)
plt.show()
print("Done!")